## 1. Imports et Configuration

In [ ]:
%matplotlib inline

import sys
import os
import joblib

# Ajouter le répertoire racine au path Python
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Import des transformateurs V3
from src.features.St_Pipeline.Transformateurs import (
    # Loaders
    ImagePathLoader,
    TupleToDataFrame,
    
    # Preprocessing
    ImageResizer,
    ImageAugmenter,
    ImageNormalizer,
    ImageMasker,
    ImageFlattener,
    ImageStandardScaler,
    RGB_to_L,
    
    # Analyse et features
    ImageAnalyser,
    ImagePCA,
    ImageHistogram,
    
    # Utilities
    SaveTransformer,
    TrainTestSplitter,
)

print("✅ Imports réussis!")
print(f"📁 Répertoire de travail: {os.getcwd()}")

## 2. Configuration des Chemins

In [ ]:
# Chemins
ROOT_DIR = "/home/lena/DS_Covid/DS_COVID/data/raw/COVID-19_Radiography_Dataset/COVID-19_Radiography_Dataset"
MODELS_DIR = "../models"
OUTPUTS_DIR = "outputs"

# Créer les dossiers
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Vérification
if os.path.exists(ROOT_DIR):
    print(f"✅ Répertoire trouvé: {ROOT_DIR}")
    labels = [d for d in os.listdir(ROOT_DIR) 
              if os.path.isdir(os.path.join(ROOT_DIR, d, 'images'))]
    print(f"📊 Labels disponibles: {labels}")
    print(f"📊 Nombre de classes: {len(labels)}")
else:
    print(f"❌ Répertoire introuvable: {ROOT_DIR}")

## 3. Pipeline 1: Preprocessing Simple

Pipeline de base pour le preprocessing et la classification.

In [ ]:
# Pipeline simple pour classification rapide
simple_preprocessing_pipeline = Pipeline([
    ('loader', ImagePathLoader(root_dir=ROOT_DIR, verbose=True)),
    ('tuple_to_df', TupleToDataFrame(verbose=True)),
    ('analyzer', ImageAnalyser(load_images=False, verbose=True)),
    ('splitter', TrainTestSplitter(test_size=0.2, random_state=42, verbose=True)),
])

# Exécution
print("\n" + "="*60)
print("🚀 Exécution Pipeline Simple")
print("="*60)

splits = simple_preprocessing_pipeline.fit_transform(None)
X_train, y_train = splits['train']
X_test, y_test = splits['test']

print(f"\n✅ Pipeline simple terminé!")
print(f"📊 Train: {len(X_train)} | Test: {len(X_test)}")

## 4. Pipeline 2: Feature Extraction avec PCA

Pipeline optimisé avec réduction de dimension par PCA.

In [ ]:
# Feature extraction pipeline
feature_extraction_pipeline = Pipeline([
    ('analyzer', ImageAnalyser(load_images=True, verbose=True)),
    ('resizer', ImageResizer(img_size=(64, 64), verbose=True)),
    ('normalizer', ImageNormalizer(verbose=True)),
    ('gray', RGB_to_L(verbose=True)),
    ('flattener', ImageFlattener(verbose=True)),
    ('pca', ImagePCA(n_components=50, verbose=True)),
])

print("\n" + "="*60)
print("🔬 Extraction des features (PCA)")
print("="*60)

# Fit sur train
print("\n🔹 TRAIN SET:")
X_train_pca = feature_extraction_pipeline.fit_transform(X_train)

# Transform sur test
print("\n🔹 TEST SET:")
X_test_pca = feature_extraction_pipeline.transform(X_test)

print(f"\n✅ Features extraites!")
print(f"📊 Train shape: {X_train_pca.shape}")
print(f"📊 Test shape: {X_test_pca.shape}")

## 5. Entraînement du Modèle

In [ ]:
print("\n" + "="*60)
print("🤖 Entraînement RandomForest")
print("="*60)

# Modèle
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Entraînement
clf.fit(X_train_pca, y_train)

print("\n✅ Modèle entraîné!")

## 6. Évaluation

In [ ]:
print("\n" + "="*60)
print("📈 Évaluation du Modèle")
print("="*60)

# Prédictions
y_pred = clf.predict(X_test_pca)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n🎯 Accuracy: {accuracy:.2%}")

# Classification Report
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred))

## 7. Matrice de Confusion

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
labels_sorted = sorted(y_test.unique())

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=labels_sorted, yticklabels=labels_sorted,
            cbar_kws={'label': 'Count'})
plt.xlabel('Prédiction', fontsize=12)
plt.ylabel('Vérité Terrain', fontsize=12)
plt.title(f'Matrice de Confusion\nAccuracy: {accuracy:.2%}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Matrice de confusion affichée!")

## 8. Importance des Features (PCA)

In [ ]:
# Visualiser la variance expliquée par PCA
pca_transformer = feature_extraction_pipeline.named_steps['pca']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Variance cumulée
cumsum = np.cumsum(pca_transformer.pca.explained_variance_ratio_)
axes[0].plot(range(1, len(cumsum)+1), cumsum, 'b-', linewidth=2)
axes[0].axhline(y=0.95, color='r', linestyle='--', label='95% variance')
axes[0].axhline(y=0.90, color='orange', linestyle='--', label='90% variance')
axes[0].set_xlabel('Nombre de composantes')
axes[0].set_ylabel('Variance expliquée cumulée')
axes[0].set_title('Variance Expliquée par PCA')
axes[0].grid(alpha=0.3)
axes[0].legend()

# Feature importance du modèle
feature_importance = clf.feature_importances_
top_n = 20
top_indices = np.argsort(feature_importance)[-top_n:][::-1]

axes[1].barh(range(top_n), feature_importance[top_indices], color='steelblue')
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels([f'PC{i+1}' for i in top_indices])
axes[1].set_xlabel('Importance')
axes[1].set_title(f'Top {top_n} Features - RandomForest')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"\n📊 Variance totale expliquée: {cumsum[-1]:.2%}")

## 9. Sauvegarde du Pipeline pour Streamlit

In [ ]:
# Créer un pipeline complet (preprocessing + split)
complete_pipeline_for_streamlit = Pipeline([
    ('loader', ImagePathLoader(root_dir=ROOT_DIR, verbose=True)),
    ('tuple_to_df', TupleToDataFrame(verbose=True)),
    ('analyzer', ImageAnalyser(load_images=True, verbose=True)),
    ('resizer', ImageResizer(img_size=(128, 128), verbose=True)),
    ('normalizer', ImageNormalizer(verbose=True)),
    ('gray', RGB_to_L(verbose=True)),
])

# Sauvegarder
pipeline_path = os.path.join(MODELS_DIR, 'pipeline_v3_main_preprocessing.pkl')
joblib.dump(complete_pipeline_for_streamlit, pipeline_path)
print(f"✅ Pipeline sauvegardé: {pipeline_path}")

# Sauvegarder aussi le pipeline avec PCA
pipeline_pca_path = os.path.join(MODELS_DIR, 'pipeline_v3_main_pca.pkl')
joblib.dump(feature_extraction_pipeline, pipeline_pca_path)
print(f"✅ Pipeline PCA sauvegardé: {pipeline_pca_path}")

# Sauvegarder le modèle
model_path = os.path.join(MODELS_DIR, 'model_v3_randomforest.pkl')
joblib.dump(clf, model_path)
print(f"✅ Modèle sauvegardé: {model_path}")

## 10. Pipeline Alternatif: Histogrammes

In [ ]:
# Pipeline avec histogrammes au lieu de PCA
histogram_pipeline = Pipeline([
    ('analyzer', ImageAnalyser(load_images=True, verbose=True)),
    ('resizer', ImageResizer(img_size=(128, 128), verbose=True)),
    ('normalizer', ImageNormalizer(verbose=True)),
    ('gray', RGB_to_L(verbose=True)),
    ('histogram', ImageHistogram(bins=32, verbose=True)),
])

print("\n" + "="*60)
print("📊 Pipeline avec Histogrammes")
print("="*60)

# Fit transform
print("\n🔹 TRAIN SET:")
X_train_hist = histogram_pipeline.fit_transform(X_train)

print("\n🔹 TEST SET:")
X_test_hist = histogram_pipeline.transform(X_test)

print(f"\n✅ Features histogrammes extraites!")
print(f"📊 Train shape: {X_train_hist.shape}")
print(f"📊 Test shape: {X_test_hist.shape}")

# Sauvegarder
histogram_path = os.path.join(MODELS_DIR, 'pipeline_v3_main_histogram.pkl')
joblib.dump(histogram_pipeline, histogram_path)
print(f"\n✅ Pipeline histogrammes sauvegardé: {histogram_path}")

## 11. Comparaison PCA vs Histogrammes

In [ ]:
print("\n" + "="*60)
print("⚖️ Comparaison: PCA vs Histogrammes")
print("="*60)

# Entraîner modèle sur histogrammes
clf_hist = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

print("\n🔹 Entraînement sur features Histogrammes...")
clf_hist.fit(X_train_hist, y_train)
y_pred_hist = clf_hist.predict(X_test_hist)
accuracy_hist = accuracy_score(y_test, y_pred_hist)

# Résultats
print("\n📊 Résultats:")
print(f"  PCA (50 composantes):      Accuracy = {accuracy:.2%}")
print(f"  Histogrammes (32 bins):    Accuracy = {accuracy_hist:.2%}")

if accuracy > accuracy_hist:
    print(f"\n🏆 PCA est meilleur de {(accuracy - accuracy_hist)*100:.2f}%")
else:
    print(f"\n🏆 Histogrammes est meilleur de {(accuracy_hist - accuracy)*100:.2f}%")

## 12. Résumé Final

In [ ]:
print("\n" + "="*60)
print("📝 RÉSUMÉ FINAL")
print("="*60)

print(f"\n📊 Dataset:")
print(f"   - Total images: {len(X_train) + len(X_test)}")
print(f"   - Train: {len(X_train)} ({len(X_train)/(len(X_train)+len(X_test))*100:.0f}%)")
print(f"   - Test: {len(X_test)} ({len(X_test)/(len(X_train)+len(X_test))*100:.0f}%)")
print(f"   - Classes: {len(labels)}")

print(f"\n🎯 Performances:")
print(f"   - PCA + RandomForest:        {accuracy:.2%}")
print(f"   - Histogram + RandomForest:  {accuracy_hist:.2%}")

print(f"\n💾 Fichiers sauvegardés:")
print(f"   - pipeline_v3_main_preprocessing.pkl")
print(f"   - pipeline_v3_main_pca.pkl")
print(f"   - pipeline_v3_main_histogram.pkl")
print(f"   - model_v3_randomforest.pkl")

print(f"\n✅ Pipeline V3 Main terminé avec succès!")
print(f"🚀 Prêt pour Streamlit!")